# Локальний інференс

Завантаження чекпоінта `result/mine_fold0_s2_e013_val0.3037.ckpt`, 5 записів з `data/train.csv`, для яких у `data/` є відповідні EEG parquet. Порівняння нормалізованих голосів експертів із передбаченими ймовірностями.

In [ ]:
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.signal import butter, sosfilt
import pyarrow.parquet as pq
import timm

# Корінь репозиторію (ноутбук у корені або в model/)
_here = Path.cwd().resolve()
if (_here / "data" / "train.csv").is_file():
    REPO = _here
elif (_here.parent / "data" / "train.csv").is_file():
    REPO = _here.parent
else:
    raise FileNotFoundError("Не знайдено data/train.csv — відкрий ноутбук з кореня hmsv2.")

DATA_DIR = REPO / "data"
CKPT_PATH = REPO / "result" / "mine_fold0_s2_e013_val0.3037.ckpt"

device = (
    torch.device("cuda") if torch.cuda.is_available() else
    torch.device("mps") if torch.backends.mps.is_available() else
    torch.device("cpu")
)
print("REPO:", REPO)
print("device:", device)
print("ckpt exists:", CKPT_PATH.is_file())

In [ ]:
VOTE_COLS = ["seizure_vote", "lpd_vote", "gpd_vote", "lrda_vote", "grda_vote", "other_vote"]
CLASS_NAMES = ["seizure", "lpd", "gpd", "lrda", "grda", "other"]
LABEL_SMOOTHING = 0.005  # як у тренуванні (dataset.py)

FS = 200
WIN_SAMPLES = 10_000
MICRO = 10
N_CH = 16
N_MACRO = WIN_SAMPLES // MICRO
CROP_LENGTHS = [2000, 5000, 10_000]

DOUBLE_BANANA = [
    ("Fp1", "F7"), ("F7", "T3"), ("T3", "T5"), ("T5", "O1"),
    ("Fp2", "F8"), ("F8", "T4"), ("T4", "T6"), ("T6", "O2"),
    ("Fp1", "F3"), ("F3", "C3"), ("C3", "P3"), ("P3", "O1"),
    ("Fp2", "F4"), ("F4", "C4"), ("C4", "P4"), ("P4", "O2"),
]
EEG_COLS = list(dict.fromkeys(c for pair in DOUBLE_BANANA for c in pair))
LR_FLIP = [4, 5, 6, 7, 0, 1, 2, 3, 12, 13, 14, 15, 8, 9, 10, 11]

_SOS = butter(5, [0.5, 20.0], btype="bandpass", fs=FS, output="sos")


def load_signal(eeg_id: int, offset_sec: float, eeg_dir: Path) -> np.ndarray:
    raw = pq.read_table(eeg_dir / f"{eeg_id}.parquet", columns=EEG_COLS).to_pandas()
    start = int(float(offset_sec) * FS)
    chunk = raw.iloc[start : start + WIN_SAMPLES]
    sig = np.stack(
        [chunk[a].values - chunk[b].values for a, b in DOUBLE_BANANA], axis=0
    ).astype(np.float32)
    if sig.shape[1] < WIN_SAMPLES:
        sig = np.pad(sig, ((0, 0), (0, WIN_SAMPLES - sig.shape[1])))
    sig = np.nan_to_num(sig, nan=0.0, posinf=1024.0, neginf=-1024.0)
    sig = np.clip(sig, -1024.0, 1024.0) / 32.0
    return sosfilt(_SOS, sig, axis=-1).astype(np.float32)


def signals_to_image(sig: np.ndarray, center: int | None = None) -> np.ndarray:
    IMG_H = N_CH * MICRO
    T = sig.shape[1]
    if center is None:
        center = T // 2
    channels = []
    for crop_len in CROP_LENGTHS:
        if crop_len >= T:
            crop = sig
        else:
            s = int(np.clip(center - crop_len // 2, 0, T - crop_len))
            crop = sig[:, s : s + crop_len]
        n = (crop.shape[1] // MICRO) * MICRO
        frame = (
            crop[:, :n]
            .reshape(N_CH, n // MICRO, MICRO)
            .transpose(0, 2, 1)
            .reshape(IMG_H, n // MICRO)
            .astype(np.float32)
        )
        if frame.shape[1] != N_MACRO:
            frame = cv2.resize(frame, (N_MACRO, IMG_H), interpolation=cv2.INTER_LINEAR)
        channels.append(frame)
    img = np.stack(channels)
    return (img - img.mean()) / (img.std() + 1e-6)

In [ ]:
class EEGNet(nn.Module):
    def __init__(self, backbone="efficientnet_b5", dropout=0.5):
        super().__init__()
        self.net = timm.create_model(
            backbone, pretrained=False, in_chans=3, num_classes=0, global_pool=""
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(self.net.num_features, 6)

    def forward(self, x):
        x = self.net(x)
        x = self.pool(x).view(x.size(0), -1)
        x = self.drop(x)
        return F.log_softmax(self.fc(x), dim=1)


FALLBACK_BACKBONE = "efficientnet_b5"


def load_model(ckpt_path: Path):
    raw = torch.load(ckpt_path, map_location=device, weights_only=True)
    if isinstance(raw, dict) and "state_dict" in raw:
        backbone = raw.get("backbone", FALLBACK_BACKBONE)
        dropout = raw.get("dropout", 0.5)
        state = raw["state_dict"]
    else:
        backbone, dropout, state = FALLBACK_BACKBONE, 0.5, raw
    model = EEGNet(backbone, dropout).to(device)
    model.load_state_dict(state)
    model.eval()
    print(f"loaded [{backbone}] dropout={dropout}")
    return model


model = load_model(CKPT_PATH)

In [ ]:
# EEG id з наявних parquet у data/ (у цьому репо їх 5)
parquet_ids = sorted(int(Path(p).stem) for p in glob.glob(str(DATA_DIR / "*.parquet")))
print("parquet eeg_id:", parquet_ids)

full_train = pd.read_csv(DATA_DIR / "train.csv")
sub = full_train[full_train["eeg_id"].isin(parquet_ids)].copy()
if sub.empty:
    raise RuntimeError("У train.csv немає рядків для цих eeg_id — перевір data/.")

# Один вікно на кожен EEG (перший eeg_sub_id), щоб було рівно 5 семплів
sub = sub.sort_values(["eeg_id", "eeg_sub_id"]).groupby("eeg_id", as_index=False).first()
sub = sub.sort_values("eeg_id").reset_index(drop=True)
print("rows для інференсу:", len(sub))
display_cols = [
    "eeg_id",
    "eeg_sub_id",
    "eeg_label_offset_seconds",
    "patient_id",
    "expert_consensus",
] + VOTE_COLS
sub[display_cols]

In [ ]:
class LocalEEGDataset(Dataset):
    """Оригінал + LR-flip для TTA (як inference_mine)."""

    def __init__(self, df: pd.DataFrame, eeg_dir: Path):
        self.df = df.reset_index(drop=True)
        self.eeg_dir = eeg_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sig = load_signal(int(row["eeg_id"]), float(row["eeg_label_offset_seconds"]), self.eeg_dir)
        img_orig = signals_to_image(sig)
        img_flip = signals_to_image(sig[LR_FLIP])
        return torch.from_numpy(img_orig), torch.from_numpy(img_flip)


loader = DataLoader(
    LocalEEGDataset(sub, DATA_DIR),
    batch_size=len(sub),
    shuffle=False,
    num_workers=0,
)

with torch.no_grad():
    for img, img_flip in loader:
        img = img.to(device)
        img_flip = img_flip.to(device)
        p = (model(img).exp() + model(img_flip).exp()) / 2.0
        pred_probs = p.cpu().numpy()

print("pred_probs shape:", pred_probs.shape)

In [ ]:
def row_target_distribution(row) -> np.ndarray:
    v = row[VOTE_COLS].values.astype(np.float64)
    v = v + LABEL_SMOOTHING
    return v / v.sum()


rows_out = []
for i, (_, row) in enumerate(sub.iterrows()):
    tgt = row_target_distribution(row)
    pr = pred_probs[i]
    ia, ip = int(tgt.argmax()), int(pr.argmax())
    log_pr = np.log(pr + 1e-12)
    kl = float((tgt * (np.log(tgt + 1e-12) - log_pr)).sum())
    rows_out.append(
        {
            "eeg_id": row["eeg_id"],
            "offset_s": row["eeg_label_offset_seconds"],
            "expert_consensus (CSV)": row["expert_consensus"],
            "true_argmax": CLASS_NAMES[ia],
            "pred_argmax": CLASS_NAMES[ip],
            "consensus==pred": str(row["expert_consensus"]).lower() == CLASS_NAMES[ip],
            "argmax_match": ia == ip,
            "KL(true||pred)": round(kl, 4),
        }
    )

summary = pd.DataFrame(rows_out)
print("Порівняння консенсусу / argmax")
display(summary)

# Детальна таблиця: голоси та ймовірності
blocks = []
for i, (_, row) in enumerate(sub.iterrows()):
    tgt = row_target_distribution(row)
    pr = pred_probs[i]
    df_i = pd.DataFrame(
        {
            "class": CLASS_NAMES,
            "votes_raw": row[VOTE_COLS].values,
            "true_prob": tgt,
            "pred_prob": pr,
        }
    )
    df_i.insert(0, "eeg_id", row["eeg_id"])
    blocks.append(df_i)

detail_long = pd.concat(blocks, ignore_index=True)
print("\nПовне порівняння по класах (true_prob зі згладжуванням як у тренуванні):")
display(detail_long)